# ***Input Embeddings*** 

<p align="center">
  <img src="./images/Input-Emdeddings.png" alt="Input Embeddings">
</p>

### What is an Input Embedding?

Input embedding converts each input token into a vector of a fixed dimension.

For example, in the original Transformer, each token is represented as a **512-dimensional vector**.
Example :

***Example :***

```text
"I love AI"
	  ↓
Tokenization
	  ↓
[I, love, AI]
	  ↓
Input Embedding
	  ↓
[512-d vector, 512-d vector, 512-d vector]
```

Each token gets its own 512-dimensional vector.

In [ ]:
import torch 
import torch.nn as nn 
import math

In [7]:
class InputEmbeddings(nn.Module):

	def __init__(self, d_model: int, vocab_size: int) -> None:
		super().__init__()
		self.d_model = d_model
		self.vocab_size = vocab_size
		self.embedding = nn.Embedding(vocab_size, d_model)

	def forward(self, x):
		# (batch, seq_len) --> (batch, seq_len, d_model)
		# Multiply by sqrt(d_model) to scale the embeddings according to the paper
		return self.embedding(x) * math.sqrt(self.d_model)


# **Positional Encoding**

<p align="center">
  <img src="./images/Positional-Encoding.png.webp" alt="Positional Encoding">
</p>

<div align="center">

We saw before that the **Embedding layer** converts each token into a vector.

For example, with a `512`-dimensional embedding:

```text
"I love AI"

     ↓

Embedding

     ↓

[
  I     → vector of size 512
  love  → vector of size 512
  AI    → vector of size 512
]
```

### **The Problem**

The Transformer does not know the **position** of each token in the sentence.

```text
I     → position 0
love  → position 1
AI    → position 2
```

### **The Solution**

We add a **Positional Encoding vector** to each token embedding.

The positional vector has the **same size** as the embedding: `512`.

```text
Token Embedding          Positional Encoding

     (512)                     (512)

        ↓                         ↓

        └────────── + ───────────┘

               ↓

             Final vector (512)
```

So:

$$
\text{Input} =
\text{Token Embedding} +
\text{Positional Encoding}
$$

### **Why do we need it?**

It gives the Transformer information about **where each token is located in the sequence**.

**Embedding** → What is the token?

**Positional Encoding** → Where is the token?

</div>


In [ ]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # Create a matrix of shape (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)
        # Create a vector of shape (seq_len)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1) # (seq_len, 1) || .unsqueeze() so we can have each number on it's own list like [[0],[1],[2],[3],....]
        # Create a vector of shape (d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # (d_model / 2)

        # Apply sine to even indices
        pe[:, 0::2] = torch.sin(position * div_term) # sin(position * (10000 ** (2i / d_model))
        # Apply cosine to odd indices
        pe[:, 1::2] = torch.cos(position * div_term) # cos(position * (10000 ** (2i / d_model))
        # Add a batch dimension to the positional encoding
        pe = pe.unsqueeze(0) # (1, seq_len, d_model)
        # Register the positional encoding as a buffer
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False) # (batch, seq_len, d_model)
        return self.dropout(x)

### 🔑 Key Notes

The original Transformer creates Positional Encoding using **sine and cosine functions**.

$$
PE(pos,2i)=
\sin\left(
\frac{pos}{10000^{2i/d_{model}}}
\right)
$$

$$
PE(pos,2i+1)=
\cos\left(
\frac{pos}{10000^{2i/d_{model}}}
\right)
$$

**Important:**

* `position` → tells us the position of each token.
* `sin` → used for even dimensions: `0, 2, 4, ...`
* `cos` → used for odd dimensions: `1, 3, 5, ...`
* Each position gets a different vector.
* The Positional Encoding has the same size as the embedding.
* Finally, we add it to the token embeddings.

$$
\boxed{
\text{Input}
=
\text{Embedding}
+
\text{Positional Encoding}
}
$$

> 💡 You don't need to memorize the formula yet. The main idea is that **sin/cos are used to create a unique vector for each position**.


# ***Layer Normalization***

<p align="center">
  <img src="./images/Layer-Normalization.png" alt="Positional Encoding">
</p>


*Layer Normalization* (**LayerNorm**) normalizes the activations of each token to make the values more stable during training.

For example:

```text
Before LayerNorm:
[0.2, 8.5, -3.1, 12.7, ...]

        ↓ LayerNorm

After LayerNorm:
[normalized values]
```

###  **Formula**

First, calculate the mean and variance:

$$
\mu = \text{mean}(x)
$$

$$
\sigma^2 = \text{variance}(x)
$$

Then normalize:

$$
\hat{x} =
\frac{x-\mu}
{\sqrt{\sigma^2+\epsilon}}
$$

Finally, apply learnable parameters:

$$
y = \gamma\hat{x}+\beta
$$

### **Key Notes**

* **LayerNorm** normalizes the activations of each token.
* It works on the **last dimension** (`d_model`).
* `γ` and `β` are **learnable parameters**.
* `ε` is a small value used to avoid division by zero.
* It helps make **training more stable**.

In PyTorch:

```python
nn.LayerNorm(d_model)
```

If:

```text
x.shape = (batch, seq_len, d_model)
```

LayerNorm is applied to:

```text
(batch, seq_len, d_model)
                    ↑
              normalize here
```

> 💡 **Main idea:** LayerNorm keeps the activations more stable so the Transformer can train more effectively.


In [8]:
class LayerNormalization(nn.Module):

    def __init__(self, features: int, eps:float=10**-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features)) # alpha is a learnable parameter
        self.bias = nn.Parameter(torch.zeros(features)) # bias is a learnable parameter

    def forward(self, x):
        # x: (batch, seq_len, hidden_size)
         # Keep the dimension for broadcasting
        mean = x.mean(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # Keep the dimension for broadcasting
        std = x.std(dim = -1, keepdim = True) # (batch, seq_len, 1)
        # eps is to prevent dividing by zero or when std is very small
        return self.alpha * (x - mean) / (std + self.eps) + self.bias